# Turismo de solo y playa

### Datos

Lo primero queremos ver cuales son los nombres de las variables y saber si la API funciona correctamente. También vemos en que formato estan los datos.

In [9]:
import requests
import pandas as pd

URL = "https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/2078"

r = requests.get(URL, params={"nult": 2}, timeout=60)
r.raise_for_status()
raw = r.json()

nombres = pd.DataFrame({
    "COD": [s["COD"] for s in raw],
    "Nombre": [s["Nombre"] for s in raw],
})

mask = (
    nombres["Nombre"].str.contains("Pernoctaciones", case=False)
    & nombres["Nombre"].str.contains("extranjero", case=False)
)
pern = nombres[mask]
nac = pern[pern["Nombre"].str.startswith("Nacional")]

print("Series totales:", len(nombres))
print("Pernoctaciones de residentes en el extranjero:", len(pern))
print("De ellas, 'Nacional.':", len(nac))
print()

for destino in ["Benidorm", "Palma", "Marbella", "Bartolom", "Adeje"]:
    print(destino)
    print(nac[nac["Nombre"].str.contains(destino, case=False)][["COD", "Nombre"]].to_string())
    print()

Series totales: 552
Pernoctaciones de residentes en el extranjero: 138
De ellas, 'Nacional.': 97

Benidorm
         COD                                                            Nombre
139  EOT2626  Nacional. Pernoctaciones. Benidorm. Residentes en el extranjero.

Palma
         COD                                                                              Nombre
191  EOT2658           Nacional. Pernoctaciones. Palma De Mallorca. Residentes en el extranjero.
431  EOT2810  Nacional. Pernoctaciones. Las Palmas De Gran Canaria. Residentes en el extranjero.

Marbella
         COD                                                            Nombre
367  EOT2762  Nacional. Pernoctaciones. Marbella. Residentes en el extranjero.

Bartolom
         COD                                                                             Nombre
435  EOT2814  Nacional. Pernoctaciones. San Bartolomé De Tirajana. Residentes en el extranjero.

Adeje
         COD                                                

In [10]:
from pathlib import Path
import requests
import pandas as pd

BASE = "https://servicios.ine.es/wstempus/js/ES/DATOS_SERIE/"

SERIES = {
    "EOT2626": "Benidorm",
    "EOT2658": "Palma de Mallorca",
    "EOT2762": "Marbella",
    "EOT2814": "San Bartolomé de Tirajana",
    "EOT2830": "Adeje",
}

Path("data").mkdir(exist_ok=True)

filas = []
for cod, destino in SERIES.items():
    r = requests.get(BASE + cod, params={"nult": 400}, timeout=60)
    r.raise_for_status()
    for d in r.json()["Data"]:
        filas.append({
            "destino": destino,
            "cod": cod,
            "fecha_ms": d["Fecha"],
            "valor": d["Valor"],
            "secreto": d["Secreto"],
            "tipo_dato": d["FK_TipoDato"],
        })

df = pd.DataFrame(filas)
df["fecha"] = (
    pd.to_datetime(df["fecha_ms"], unit="ms", utc=True)
    .dt.tz_convert("Europe/Madrid")
    .dt.tz_localize(None)
    .dt.to_period("M")
    .dt.to_timestamp()
)
df = df.drop(columns="fecha_ms").sort_values(["destino", "fecha"])
df["descargado"] = pd.Timestamp.today().normalize()

df.to_parquet("data/pernoctaciones_raw.parquet", index=False)

# ¿El tipo 2 son siempre los mismos meses en los 5 destinos?
print(df[df["tipo_dato"] == 2].groupby("destino")["fecha"].agg(["min", "max", "count"]))

                                 min        max  count
destino                                               
Adeje                     2026-01-01 2026-09-01      9
Benidorm                  2026-01-01 2026-09-01      9
Marbella                  2026-01-01 2026-09-01      9
Palma de Mallorca         2026-01-01 2026-09-01      9
San Bartolomé de Tirajana 2026-01-01 2026-09-01      9


In [11]:
# Qué mes falta en cada destino (frente al calendario completo)
calendario = pd.date_range("2005-01-01", "2026-09-01", freq="MS")
for destino, g in df.groupby("destino"):
    faltan = calendario.difference(g["fecha"])
    print(destino, "-> meses que faltan:", list(faltan.strftime("%Y-%m")))

# Valores nulos y ceros
print()
print("Nulos por destino:")
print(df[df["valor"].isna()].groupby("destino")["fecha"].apply(lambda s: list(s.dt.strftime("%Y-%m"))))
print()
print("Ceros por destino:")
print(df[df["valor"] == 0].groupby("destino")["fecha"].apply(lambda s: list(s.dt.strftime("%Y-%m"))))

# Entorno del COVID
print()
print(df[(df["fecha"] >= "2020-02-01") & (df["fecha"] <= "2020-07-01")]
      .pivot(index="fecha", columns="destino", values="valor"))

Adeje -> meses que faltan: []
Benidorm -> meses que faltan: []
Marbella -> meses que faltan: ['2020-05']
Palma de Mallorca -> meses que faltan: []
San Bartolomé de Tirajana -> meses que faltan: []

Nulos por destino:
destino
Adeje                                 [2020-05, 2020-06]
Benidorm                     [2020-05, 2020-06, 2021-02]
Marbella                                       [2020-06]
Palma de Mallorca                     [2020-05, 2020-06]
San Bartolomé de Tirajana             [2020-05, 2020-06]
Name: fecha, dtype: object

Ceros por destino:
destino
Adeje                        [2020-04]
Benidorm                     [2020-04]
Marbella                     [2020-04]
Palma de Mallorca            [2020-04]
San Bartolomé de Tirajana    [2020-04]
Name: fecha, dtype: object

destino        Adeje  Benidorm  Marbella  Palma de Mallorca  \
fecha                                                         
2020-02-01  823845.0  312054.0  107632.0           207239.0   
2020-03-01  358033.0  2

In [12]:
# Benidorm, febrero 2021: ¿secreto estadístico?
print(df[(df["destino"] == "Benidorm") & (df["fecha"].between("2021-01-01", "2021-03-01"))])
print()
print("Datos marcados como secretos por destino:")
print(df.groupby("destino")["secreto"].sum())

# Calendario completo: una fila por mes, una columna por destino
calendario = pd.date_range("2005-01-01", "2026-09-01", freq="MS")
completo = df.set_index(["destino", "fecha"])["valor"].unstack("destino").reindex(calendario)
completo.index.name = "fecha"

print()
print("Tamaño:", completo.shape)
print(completo.isna().sum())
print()
print(completo.loc["2020-03":"2020-07"])

      destino      cod   valor  secreto  tipo_dato      fecha descargado
192  Benidorm  EOT2626  2484.0    False          1 2021-01-01 2026-11-20
193  Benidorm  EOT2626     NaN     True          1 2021-02-01 2026-11-20
194  Benidorm  EOT2626  2006.0    False          1 2021-03-01 2026-11-20

Datos marcados como secretos por destino:
destino
Adeje                        2
Benidorm                     3
Marbella                     1
Palma de Mallorca            2
San Bartolomé de Tirajana    2
Name: secreto, dtype: int64

Tamaño: (261, 5)
destino
Adeje                        2
Benidorm                     3
Marbella                     2
Palma de Mallorca            2
San Bartolomé de Tirajana    2
dtype: int64

destino        Adeje  Benidorm  Marbella  Palma de Mallorca  \
fecha                                                         
2020-03-01  358033.0  235608.0   61237.0            97316.0   
2020-04-01       0.0       0.0       0.0                0.0   
2020-05-01       NaN       

In [13]:
sec = df[df["secreto"]].copy()
sec["fecha"] = sec["fecha"].dt.strftime("%Y-%m")
print(sec[["destino", "fecha", "valor"]].to_string(index=False))

                  destino   fecha  valor
                    Adeje 2020-05    NaN
                    Adeje 2020-06    NaN
                 Benidorm 2020-05    NaN
                 Benidorm 2020-06    NaN
                 Benidorm 2021-02    NaN
                 Marbella 2020-06    NaN
        Palma de Mallorca 2020-05    NaN
        Palma de Mallorca 2020-06    NaN
San Bartolomé de Tirajana 2020-05    NaN
San Bartolomé de Tirajana 2020-06    NaN


In [14]:
r = requests.get(BASE + "EOT2626", params={"nult": 400}, timeout=60)
r.raise_for_status()
tipos = pd.DataFrame(r.json()["Data"])

print(tipos["FK_TipoDato"].value_counts())
print()
print(tipos[["Anyo", "FK_Periodo", "FK_TipoDato"]].tail(10))

FK_TipoDato
1    252
2      9
Name: count, dtype: int64

     Anyo  FK_Periodo  FK_TipoDato
251  2025          12            1
252  2026           1            2
253  2026           2            2
254  2026           3            2
255  2026           4            2
256  2026           5            2
257  2026           6            2
258  2026           7            2
259  2026           8            2
260  2026           9            2


### Particion